In [ ]:
!pip install transformers==4.57.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [ ]:
# introduce gender tags
special_tokens_dict = {'additional_special_tokens': ['<female>', '<male>', '<neutral>']}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


MBartScaledWordEmbedding(250057, 1024, padding_idx=1)

In [ ]:
# testing sentence
eng_sentence = "this is a test"

# translate eng test sentence to ukr
tokenizer.src_lang = "en_XX"
encoded_hi = tokenizer(eng_sentence, return_tensors="pt")
generated_tokens = model.generate(
    **encoded_hi,
    forced_bos_token_id=tokenizer.lang_code_to_id["uk_UA"]
)
tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

['це тест']

In [ ]:
import pandas as pd

# dataframe with tagged sentences
df = pd.read_csv('/content/tatoeba_full_tagged.csv')

lst_eng_sentences = df['Tagged EN'].dropna().tolist()
lst_ukr_sentences = df['ukr_reference'].dropna().tolist()


Data loading and tokenization

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset
import pandas as pd


df = pd.read_csv("/content/tatoeba_full_tagged.csv")
df = df.dropna(subset=["Tagged EN", "ukr_reference"])
df = df[df["Tagged EN"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
df = df[df["ukr_reference"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
print(f"Loaded {len(df)} sentence pairs after cleaning.")


model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)



Loaded 1285 sentence pairs after cleaning.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Intriduce tags

In [ ]:

special_tokens_dict = {"additional_special_tokens": ["<female>", "<male>", "<neutral>"]}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))


tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "uk_UA"
model.config.decoder_start_token_id = tokenizer.lang_code_to_id["uk_UA"]



The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


preprocessing

In [ ]:
train_data = Dataset.from_dict({
    "src": df["Tagged EN"].tolist(),
    "tgt": df["ukr_reference"].tolist()
})

def preprocess_function(batch):
    srcs = batch["src"]
    tgts = batch["tgt"]

    return tokenizer(
        srcs,
        text_target=tgts,
        max_length=128,
        truncation=True,
    )

tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names,
    desc="Tokenizing dataset",
)

print(f"Tokenization done. {len(tokenized_train)} examples ready.")



Tokenizing dataset:   0%|          | 0/1285 [00:00<?, ? examples/s]

Tokenization done. 1285 examples ready.


Training setup

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding="longest")

training_args = Seq2SeqTrainingArguments(
    output_dir="./mbart_gender_finetune",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    predict_with_generate=True,
    save_total_limit=1,
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

#train model
trainer.train()

#save model
trainer.save_model("./mbart_gender_finetune")
tokenizer.save_pretrained("./mbart_gender_finetune")

print("Training complete. Model saved to ./mbart_gender_finetune")

/tmp/ipykernel_27097/1696322278.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,1.289800
20,0.888000
30,1.092800
40,1.262400
50,0.893600
60,0.996100
70,1.159100
80,1.154800
90,1.378400
100,0.998800


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Training complete. Model saved to ./mbart_gender_finetune


In [ ]:
!zip -r /content/drive/MyDrive/mbart_gender_finetune.zip /content/mbart_gender_finetune

  adding: content/mbart_gender_finetune/ (stored 0%)
  adding: content/mbart_gender_finetune/training_args.bin (deflated 53%)
  adding: content/mbart_gender_finetune/sentencepiece.bpe.model (deflated 49%)
  adding: content/mbart_gender_finetune/checkpoint-966/ (stored 0%)
  adding: content/mbart_gender_finetune/checkpoint-966/optimizer.pt (deflated 8%)
  adding: content/mbart_gender_finetune/checkpoint-966/training_args.bin (deflated 53%)
  adding: content/mbart_gender_finetune/checkpoint-966/rng_state.pth (deflated 26%)
  adding: content/mbart_gender_finetune/checkpoint-966/trainer_state.json (deflated 79%)
  adding: content/mbart_gender_finetune/checkpoint-966/sentencepiece.bpe.model (deflated 49%)
  adding: content/mbart_gender_finetune/checkpoint-966/model.safetensors (deflated 7%)
  adding: content/mbart_gender_finetune/checkpoint-966/config.json (deflated 60%)
  adding: content/mbart_gender_finetune/checkpoint-966/special_tokens_map.json (deflated 73%)
  adding: content/mbart_gen

In [ ]:
# from google.colab import files
# files.download("/content/mbart_gender_finetune.zip")

Test

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [ ]:

from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import pandas as pd

model.to("cuda")
model.half()

# Load model
model_path = "./mbart_gender_finetune"

tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
model = MBartForConditionalGeneration.from_pretrained(model_path)

tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "uk_UA"

# Start decoder with Ukrainian token
forced_bos_id = tokenizer.lang_code_to_id["uk_UA"]

#Load data
df_test = pd.read_csv("/content/test_tagged.csv")
test_sentences = df_test["source"].tolist()

print(f"Loaded {len(test_sentences)} test sentences.")

# Translate!
outputs = []

for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_id,
        max_length=128
    )

    translation = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

    outputs.append({"EN": sentence, "UKR": translation})

# save translations
df_out = pd.DataFrame(outputs)
df_out.to_csv("mbart_tagged_translations.csv", index=False)

print("Saved test translations to mbart_tagged_translations.csv")


Loaded 738 test sentences.
Saved test translations to mbart_tagged_translations.csv


In [ ]:
df_out.to_csv("/content/drive/MyDrive/mbart_tagged_translation.tsv", index=False, sep="\t")